# Member 1 — Full Reproduction: Structured Forum Signals → Continuous Stress Regression

**Role:** tabular / structured-data specialist. Member 1 predicts the continuous `final_stress`
score from metadata and precomputed numeric signals, then hands leakage-safe OOF and
validation/test ensemble predictions — together with the exact engineered features — to
Member C's final late-fusion model.

## Scientific role of this branch

Member 1 answers a different question from Member 2:

> **How much stress-related information can be learned from structured forum context,
> behavioral variables, timing, metadata, and interpretable count signals without modeling
> the raw Persian post as free text?**

This separation is intentional. Member 2 learns contextual meaning directly from Persian
text with ParsBERT. Member 1 instead uses structured signals such as post length,
punctuation, emoji counts, positive/negative lexical counts, starter/reply status, account
activity, dates, category, sub-category, education, and gender. Keeping the branches
separate creates complementary model outputs that can later be combined by Member C.

The target is a **continuous stress score**. Regression preserves severity distance:
predicting 6.8 for a target of 7.0 is a smaller error than predicting 2.0, even if both can
be discussed through operational classes later. The four classes are used for evaluation
and threshold-based reporting; they are not the direct CatBoost training objective.

## Why CatBoost?

CatBoost is a gradient-boosted decision-tree algorithm designed for heterogeneous tabular
data. Boosting builds trees sequentially; each new tree learns patterns in the residual
errors left by the previous trees. This allows nonlinear effects and interactions, for
example:

- negative-language counts may matter differently for starter posts than replies;
- a raw count can have a different meaning in a very short post than in a long post;
- posting time, category, and account activity can interact;
- missing metadata can itself carry information.

CatBoost is particularly suitable here because it:

1. handles numerical and categorical features in one model;
2. supports missing numerical values without mean imputation;
3. uses ordered categorical statistics / ordered boosting mechanisms designed to reduce
   target leakage in category encoding;
4. performs well on small-to-medium tabular datasets without a large neural network; and
5. provides native feature-importance and SHAP functionality for post-hoc explanation.

## Leakage-safe modeling design

The final data contract contains 4,226 training rows, 452 validation rows, 453 locked-test
rows, and 484 embargo rows. Related rows can share authors, threads, or duplicated content,
so ordinary row-random splitting would be optimistic. The supplied project-wide grouping
keeps related rows inside one role and one training fold.

For fusion, training predictions must be **out-of-fold (OOF)**: each training row is
predicted by a model that did not train on that row. Validation and test are predicted by
the mean of the five fold models. This gives Member C a realistic base-model signal instead
of an artificially accurate in-sample prediction.

## What this notebook reproduces

Running the notebook from top to bottom:

1. resolves the project root and records the software environment;
2. validates the frozen handoff, roles, groups, IDs, folds, targets, and feature contract;
3. recreates exactly 67 ordered structured features;
4. validates the five supplied leakage-safe folds;
5. trains or loads five weighted CatBoost regressors;
6. generates complete OOF training predictions;
7. generates five-model validation/test ensembles;
8. exports the exact Member C handoff;
9. evaluates continuous and class-based behavior;
10. generates post-hoc feature importance and optional native SHAP;
11. verifies all artifacts and runs contract tests; and
12. writes a compact final reproduction summary.

> **Safety and scope:** this is a research stress-risk model, not a clinical diagnosis.
> The class recall floors are internal project objectives, not medical standards.
> The tabular branch is a complementary base model; the final project result belongs to
> the frozen Member C fusion system.


## 1. Environment and project setup

### What this block does

This block finds the project root without depending on the literal ZIP/folder name, adds
the root to Python's import path, loads the project/model configuration, reads execution
flags, sets reproducibility seeds, creates output directories, records package versions,
and starts a run log.

### Why use configuration files instead of hard-coding values?

The role names, paths, feature contract, folds, prediction thresholds, model
hyperparameters, and output locations are part of the experiment definition. Keeping them
in JSON files makes the notebook easier to audit and prevents slightly different values
from being copied into different cells.

### Two execution modes

- `FULL_RETRAIN=True`: fit all five fold models and the optional full-train model again.
- `FULL_RETRAIN=False`: load the included native `.cbm` files and reproduce predictions,
  metrics, figures, and handoff artifacts.

Loading is the safest exact-reproduction route. Retraining is useful to demonstrate the
algorithm, but tiny floating-point differences can occur across CPUs, thread scheduling,
and library versions. CatBoost's native `.cbm` format is used because it is more portable
than pickling a live Python model object.

### Why set seeds and record versions?

Boosted trees still use randomized operations such as feature/random-strength behavior and
bootstrap sampling. Setting the project seed makes each fold's random state explicit:
`seed = 42 + fold`. Package/version recording provides provenance. A seed reduces
avoidable randomness but cannot guarantee byte-identical floating-point results on every
hardware/software environment.

### Why CPU?

The final tabular data are small enough for CatBoost CPU training to complete in minutes.
GPU complexity is unnecessary, and CPU `.cbm` models are straightforward to reproduce on
ordinary laptops.


In [1]:
from pathlib import Path
import os, sys, json, logging

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "configs" / "project_config.json").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Project root could not be resolved.")

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.common import load_json, save_json, set_all_seeds, package_versions, ensure_output_directories

PROJECT_CONFIG = load_json(ROOT / "configs" / "project_config.json")
MODEL_CONFIG = load_json(ROOT / "configs" / "model_config.json")
FULL_RETRAIN = os.getenv("MEMBER1_FULL_RETRAIN", str(PROJECT_CONFIG["full_retrain_default"])).lower() in {"1", "true", "yes"}
RUN_NATIVE_SHAP = os.getenv("MEMBER1_RUN_SHAP", str(PROJECT_CONFIG["run_native_shap_default"])).lower() in {"1", "true", "yes"}

set_all_seeds(PROJECT_CONFIG["random_seed"])
OUTPUT_PATHS = ensure_output_directories(ROOT, PROJECT_CONFIG)
versions = package_versions()
save_json(versions, OUTPUT_PATHS["metrics"] / "package_versions.json")

logging.basicConfig(
    filename=OUTPUT_PATHS["logs"] / "reproduction.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    force=True,
)
logging.info("Notebook reproduction started; FULL_RETRAIN=%s", FULL_RETRAIN)

print("Project root:", ROOT)
print("FULL_RETRAIN:", FULL_RETRAIN)
print("RUN_NATIVE_SHAP:", RUN_NATIVE_SHAP)
print("Compute mode:", MODEL_CONFIG["task_type"])
print("Python/package versions:")
for key, value in versions.items():
    print(f"  {key}: {value}")

Project root: /mnt/data/Member1_Final_Reproducible_Project
FULL_RETRAIN: True
RUN_NATIVE_SHAP: False
Compute mode: CPU
Python/package versions:
  python: 3.13.5
  platform: Linux-6.18.35-x86_64-with-glibc2.41
  numpy: 2.3.5
  pandas: 2.2.3
  scipy: 1.17.0
  scikit_learn: 1.8.0
  catboost: 1.2.8
  joblib: 1.5.3
  matplotlib: 3.10.8


## 2. Input validation

### What this block does

The validator loads five accepted input artifacts and checks that the handoff is internally
consistent with the global modeling manifest before any feature engineering or training
begins.

It checks, among other things:

- required columns and files exist;
- `unique_post_id` is present, non-null, and unique;
- handoff and manifest contain the same ID set;
- roles are exactly train / validation / test / embargo;
- targets, roles, folds, classes, groups, and IDs agree between files;
- only training rows have OOF fold assignments;
- training weights are finite and positive;
- no author, thread, content hash, or connected `group_id` crosses modeled roles/folds;
- output templates have the expected schema; and
- no forbidden target/text/split field appears in the 67-feature model order.

### Why fail early?

Most damaging machine-learning mistakes are silent contract errors. A model can train and
produce plausible metrics while rows are misaligned, duplicated, or leaked between
training and evaluation. Failing before fitting is therefore a scientific safeguard, not
just defensive programming.

### About the class column and boundary values

Evaluation uses the frozen `clinical_class` supplied by the data foundation as the
authoritative class label. The project-wide conceptual bins are Low ≤3, Moderate (3,5],
High (5,7], and Very High >7. Some stored continuous scores lie exactly on printed
boundaries because of historical averaging/rounding, so the package preserves the frozen
class column rather than silently relabeling rows inside Member 1. Prediction thresholds
for Member 1 are a separate validation-calibrated mapping from **model-score space** to the
four classes.

### Why is embargo validated but not modeled?

Embargo rows share provenance/component constraints that make them unsuitable for clean
training or official evaluation. They are kept in the package for completeness and
contract validation, but are never transformed, fitted, or predicted.


In [2]:
from src.validate_inputs import load_inputs, validate_inputs

INPUTS = load_inputs(ROOT, PROJECT_CONFIG)
AUDIT = validate_inputs(INPUTS, PROJECT_CONFIG)
save_json(AUDIT, OUTPUT_PATHS["metrics"] / "input_audit.json")

print("Loaded Member C handoff files:")
for name, path in INPUTS["paths"].items():
    print(f"  {name}: {path.relative_to(ROOT)}")
print("\nRole counts:", AUDIT["role_counts"])
print("Fold counts:", AUDIT["fold_counts"])
print("Feature contract passed:", AUDIT["input_contract_passed"])
print("Embargo used for modeling:", AUDIT["embargo_used"])

Loaded Member C handoff files:
  handoff: inputs/member1_handoff.csv.gz
  manifest: inputs/modeling_manifest_v2.csv
  feature_contract: inputs/member1_feature_contract.json
  oof_template: inputs/oof_predictions_template.csv
  holdout_template: inputs/holdout_predictions_template.csv

Role counts: {'train': 4226, 'embargo': 484, 'test': 453, 'validation': 452}
Fold counts: {'0': 844, '1': 844, '2': 848, '3': 847, '4': 843}
Feature contract passed: True
Embargo used for modeling: False


## 3. Data preparation and exact feature contract

### What this block does

The preparation stage separates the frozen roles, quarantines embargo, applies one
stateless `Member1FeatureEngineer` to train/validation/test, preserves one exact
67-column order, and saves the preprocessing configuration and processed-data manifest.

A **stateless** transformer is appropriate because these transformations do not estimate
means, vocabularies, or scaling parameters from the full dataset. They are deterministic
rules such as parsing dates, computing ratios, mapping missing values, and selecting
predefined columns. This reduces train/holdout representation drift.

### Feature families and reasoning

The 67 features cover:

1. **Interaction and post role:** likes, starter/reply status, and whether the post replies
   to another post.
2. **Account behavior:** user post count, account age, posts per year, children count, and
   smoothed interaction rates.
3. **Temporal context:** Jalali year/month/day/hour/minute, weekday, daypart, night/weekend
   indicators, and cyclic encodings.
4. **Forum/demographic context:** gender, category, sub-category, and cleaned education.
5. **Signature signals:** presence plus length, punctuation, emoji, and lexical counts.
6. **Post surface/lexical signals:** length, punctuation, questions, exclamations, emojis,
   positive/negative counts, and normalized ratios.

### Why log and ratio transformations?

Raw activity variables are strongly right-skewed. `log1p` compresses very large values
while remaining defined at zero. Ratios normalize evidence by opportunity: for example,
three negative terms in a 15-word post may be more informative than three in a 500-word
post. Denominators are clipped/smoothed to avoid division by zero and unstable infinities.

### Why sine/cosine for hour and month?

Clock time and month are circular: hour 23 is close to hour 0, but ordinary integers make
them look far apart. Sine/cosine pairs preserve this wrap-around geometry while retaining
the original hour/month features for tree splits.

### Missing-value policy

- invalid/missing age is represented as numeric NaN plus an `age_missing` indicator;
- CatBoost receives numeric NaNs directly;
- missing categorical values use `__MISSING__`;
- historical signature value `-1` means “no signature,” so signature counts become zero
  and `has_signature=0`;
- dates that cannot be parsed produce missing derived values rather than fabricated dates.

### Why raw text is forbidden here

`content` and `thread_title` are never passed to CatBoost as free text. The post/signature
count columns are upstream numerical measurements produced by the early data pipeline.
`stress_proxy`, `final_stress`, `clinical_class`, IDs, roles, folds, weights, and annotation
provenance are forbidden features. This prevents target leakage and preserves Member 1 as
a genuinely structured branch.

### Important inference limitation

The original lexical resources used to reconstruct every predefined positive/negative
count from a completely raw new post are not included in this Member 1 package. Therefore,
new-post inference requires the upstream service to supply the same metadata/count columns.
The package fails clearly rather than inventing replacement counts.


In [3]:
from src.prepare_data import prepare_data

PREPARED = prepare_data(INPUTS["handoff"], ROOT, PROJECT_CONFIG)
print("Rows prepared:", PREPARED["manifest"]["row_counts"])
print("Embargo rows quarantined and not transformed:", len(PREPARED["embargo"]))
print("Exact engineered feature count:", PREPARED["manifest"]["feature_count"])
print("Categorical features:", PREPARED["engineer"].categorical_features_)
print("\nExact feature order:")
for i, feature in enumerate(PREPARED["engineer"].feature_order_, 1):
    print(f"{i:02d}. {feature}")

Rows prepared: {'train': 4226, 'validation': 452, 'test': 453, 'embargo': 484}
Embargo rows quarantined and not transformed: 484
Exact engineered feature count: 67
Categorical features: ['gender', 'category', 'sub_category', 'education_clean', 'posted_weekday_name', 'posted_daypart']

Exact feature order:
01. likes
02. log1p_likes
03. user_post_count
04. log1p_user_post_count
05. children_count
06. age_missing
07. age_num
08. is_starter
09. has_reply
10. gender
11. category
12. sub_category
13. education_clean
14. posted_year
15. posted_month
16. posted_day
17. posted_hour
18. posted_minute
19. posted_weekday
20. posted_weekday_name
21. posted_daypart
22. posted_is_weekend
23. posted_is_night
24. posted_is_late_evening
25. posted_hour_sin
26. posted_hour_cos
27. posted_month_sin
28. posted_month_cos
29. join_year
30. join_month
31. join_day
32. account_age_days
33. account_age_years
34. posts_per_account_year
35. log1p_posts_per_account_year
36. likes_per_100_posts
37. has_signature
38

## 4. Frozen grouped-fold handling

### What this block does

The notebook uses the five fold assignments supplied by the final Member C data foundation
and saves an auditable fold table. It does **not** generate a new random split.

Final held-out fold sizes are 844, 844, 848, 847, and 843 rows.

### Why grouping is necessary

Forum rows violate the independent-and-identically-distributed assumption:

- the same author has a recognizable writing style and repeated personal context;
- posts in the same thread share conversation context;
- duplicated or near-duplicate content can appear more than once.

If connected rows crossed folds, the held-out prediction could benefit from highly similar
training examples. The project therefore created connected groups from author, thread,
and exact-content relationships and keeps every group inside one fold/role.

### Why five folds?

Five folds balance statistical quality and computation. Each model trains on roughly 80%
of the 4,226 training rows and predicts the remaining ~20%. This yields one honest
prediction for every training row while requiring only five CatBoost fits. More folds
would increase compute and correlation between training sets; fewer folds would train each
model on substantially less data.

### Why retain the supplied folds exactly?

The folds are shared by Member 1, Member 2, and Member C. Changing them locally would break
alignment between the two base-model OOF predictions and could invalidate the final fusion
experiment.


In [4]:
from src.build_folds import validate_and_save_folds

FOLD_AUDIT = validate_and_save_folds(
    PREPARED["raw"]["train"],
    INPUTS["manifest"],
    ROOT,
    PROJECT_CONFIG,
)
print("Supplied folds retained; no random split was created.")
print("Fold counts:", FOLD_AUDIT["fold_counts"])
print("Group-separation audit:", AUDIT["group_separation"])

Supplied folds retained; no random split was created.
Fold counts: {'0': 844, '1': 844, '2': 848, '3': 847, '4': 843}
Group-separation audit: {'group_id': {'cross_modeled_roles': 0, 'cross_train_folds': 0}, 'author': {'cross_modeled_roles': 0, 'cross_train_folds': 0}, 'thread_id': {'cross_modeled_roles': 0, 'cross_train_folds': 0}, 'content_hash': {'cross_modeled_roles': 0, 'cross_train_folds': 0}}


## 5. Five-fold CatBoost training or model loading

### Algorithmic intuition

Gradient boosting builds an additive model:

\[
\hat y(x)=\sum_{t=1}^{T}\eta f_t(x),
\]

where each tree \(f_t\) is fitted to reduce the residual error of the trees already built
and \(\eta\) is the learning rate. A small learning rate with many trees often generalizes
better than a few aggressive trees.

The frozen model uses:

- 558 trees;
- depth 8;
- learning rate 0.025;
- L2 leaf regularization 10;
- random strength 0.3;
- RMSE training loss; and
- the supplied `training_sample_weight`.

### Why these settings?

Depth 8 allows interactions among several structured variables without creating an
extremely high-capacity tree. The low learning rate makes each tree a modest correction.
L2 regularization shrinks leaf values and helps control overfitting. `random_strength`
adds randomness to split scoring, which can improve robustness on a small dataset.

The accepted CPU models resolve to CatBoost's MVS bootstrap behavior. The configuration
also retains `bagging_temperature` for historical completeness, but that value is not an
active parameter under MVS in the serialized final models.

### Why sample weights?

Training weights combine project-level annotation confidence/provenance and moderated
class balancing. They affect optimization only: a row with a larger weight contributes
more to the RMSE objective. Validation/test metrics remain unweighted so the reported
numbers describe the actual holdout rows.

### Fold training

For fold `k`, the model fits only rows from the other four folds and predicts fold `k`.
The held-out fold is passed as an evaluation set so learning curves can be recorded, but
the final configuration uses a fixed 558 iterations with `use_best_model=False`; the fold
is **not** used for early stopping or tree-count selection.

This distinction is important. Early stopping separately on every held-out fold would
produce fold models with different selected complexities and would use OOF labels for a
model-selection role. The fixed iteration count preserves a simpler, pre-specified
training contract.

### Why an optional full-train model?

A model trained on all 4,226 training rows is saved for completeness, but it is not used
for the official OOF/validation/test handoff. The default future Member 1 prediction is the
mean of the same five fold models used by the fusion contract, preserving representation
consistency.


In [5]:
from src.train_models import train_or_load_fold_models

FOLD_MODELS = train_or_load_fold_models(
    PREPARED,
    ROOT,
    PROJECT_CONFIG,
    MODEL_CONFIG,
    full_retrain=FULL_RETRAIN,
)
print(f"Available fold models: {len(FOLD_MODELS)}")
for fold in range(PROJECT_CONFIG["number_of_folds"]):
    path = ROOT / "models" / f"fold_{fold}" / "model.cbm"
    print(f"  fold {fold}: {path.relative_to(ROOT)} ({path.stat().st_size / 1024**2:.2f} MB)")

Available fold models: 5
  fold 0: models/fold_0/model.cbm (2.25 MB)
  fold 1: models/fold_1/model.cbm (2.24 MB)
  fold 2: models/fold_2/model.cbm (2.24 MB)
  fold 3: models/fold_3/model.cbm (2.22 MB)
  fold 4: models/fold_4/model.cbm (2.24 MB)


## 6. OOF prediction generation

### What is an out-of-fold prediction?

An OOF prediction is a prediction for a training row made by a model that did not train on
that row. For each fold:

1. train CatBoost on the other four folds;
2. predict only the held-out fold;
3. place those predictions back into their original ID positions.

Concatenating all five held-out blocks gives exactly one prediction for each of the 4,226
training rows.

### Why OOF predictions are essential for fusion

Member C trains a second-level model on the base-model outputs. If Member 1 supplied
in-sample predictions, CatBoost would appear much more accurate on the fusion-training
rows than on future data. The fusion model would learn an unrealistically strong trust in
Member 1 — a form of stacking leakage.

OOF predictions approximate the error distribution that the base model will have on unseen
rows. For each OOF row, `prediction_std_optional` is blank because exactly one held-out
model is allowed to produce that row's training prediction.

### Why predictions are clipped

The target scale is 1–10. Tree ensembles can occasionally extrapolate slightly outside
that interval, so final predictions are clipped to the valid project range before
evaluation and handoff. Clipping enforces the target domain; it does not replace model
calibration.


In [6]:
from src.generate_oof_predictions import generate_oof_predictions

OOF_OUTPUT, OOF_PREDICTION = generate_oof_predictions(
    FOLD_MODELS,
    PREPARED,
    PROJECT_CONFIG,
)
print("OOF rows:", len(OOF_OUTPUT))
print("Unique OOF IDs:", OOF_OUTPUT["unique_post_id"].nunique())
print("Missing OOF predictions:", int(OOF_OUTPUT["prediction"].isna().sum()))
display(OOF_OUTPUT.head())

OOF rows: 4226
Unique OOF IDs: 4226
Missing OOF predictions: 0


,unique_post_id,fold,true_stress,prediction,prediction_std_optional
0,246749152,0,1.000000,1.057848,NaN
1,246750589,0,5.111111,5.095725,NaN
2,246952723,0,2.000000,3.042569,NaN
3,249384092,0,2.629630,3.228702,NaN
4,251560308,0,1.871310,3.193136,NaN


## 7. Validation and locked-test five-model ensembles

Validation and test rows are outside all five training folds. Therefore, every fold model
can predict each holdout row without having trained on it.

For a holdout row \(i\), the delivered prediction is:

\[
\hat y_i=\frac{1}{5}\sum_{k=0}^{4}\hat y_{ik}.
\]

### Why average five models?

Each fold model sees a slightly different 80% training subset. Their errors are not
identical. Averaging reduces variance and dependence on one fold's sample composition. The
population standard deviation across the five predictions is also saved as
`prediction_std_optional`; it measures **between-model disagreement**, not a calibrated
probability or guaranteed uncertainty interval.

### Validation versus locked test

- **Validation** may be used for development decisions and for freezing Member 1's
  prediction-to-class thresholds.
- **Test** is opened only after the model contract and thresholds are frozen.

The notebook can reproduce test metrics repeatedly from unchanged artifacts, but no model,
feature, or threshold may be modified because of those results and then re-reported on the
same test as fresh evidence.

### Model-specific thresholds

Member 1's frozen prediction thresholds are approximately 2.929, 4.265, and 6.215. They
are not new definitions of true stress. They map the tabular model's compressed prediction
scale to the authoritative four evaluation classes and were selected before test
inspection.


In [7]:
from src.generate_validation_predictions import generate_validation_predictions
from src.generate_test_predictions import generate_test_predictions

VALIDATION_OUTPUT, VALIDATION_FOLD_OUTPUT = generate_validation_predictions(
    FOLD_MODELS, PREPARED, PROJECT_CONFIG
)
TEST_OUTPUT, TEST_FOLD_OUTPUT = generate_test_predictions(
    FOLD_MODELS, PREPARED, PROJECT_CONFIG
)

print("Validation rows:", len(VALIDATION_OUTPUT))
print("Test rows:", len(TEST_OUTPUT))
print("Validation/test prediction method: mean of five fold models")
print("Test labels were not used for training or model selection.")
display(VALIDATION_FOLD_OUTPUT.head())

Validation rows: 452
Test rows: 453
Validation/test prediction method: mean of five fold models
Test labels were not used for training or model selection.


,unique_post_id,fold_0_prediction,fold_1_prediction,fold_2_prediction,fold_3_prediction,fold_4_prediction,ensemble_prediction,prediction_std
0,253589219,3.066437,2.905617,2.920570,3.147725,3.166584,3.041387,0.110127
1,253589632,2.854865,2.722607,2.730446,2.732075,2.813477,2.770694,0.053552
2,253590070,3.352891,3.153245,3.370558,3.490749,3.315518,3.336592,0.108852
3,253590597,3.043607,2.635210,2.833111,2.864630,2.813040,2.837920,0.130195
4,256431909,3.683748,3.752370,3.964123,3.649222,3.755742,3.761041,0.109383


## 8. Member C fusion handoff export

### What Member C receives

Member C receives two complementary products from Member 1:

1. **Scalar predictions**
   - OOF predictions for all training rows;
   - five-model mean predictions for validation and test;
   - fold-disagreement standard deviation for holdouts.
2. **The exact engineered feature matrices**
   - the same 67 ordered features seen by the CatBoost branch for train/validation/test.

The final hybrid fusion can therefore use both Member 1's summarized model opinion and
selected structured context.

### Why `unique_post_id` is mandatory

CSV row order is not a scientific alignment key. Filtering or sorting can silently change
order, so all joins are performed through the canonical string `unique_post_id`. The
export verifies complete ID coverage, uniqueness, and role separation.

### Why the train and holdout files differ

Training must use held-out OOF predictions. Validation/test use five-model ensemble
predictions. Replacing OOF values with predictions from a model trained on all training
rows would leak target information into the stacker.

### What must never enter the fusion as a feature from this handoff

`true_stress`, `clinical_class`, role/fold fields, IDs, and weights are labels or contract
metadata. They are retained for auditing/evaluation but must not be used as predictors.


In [8]:
from src.export_memberC_handoff import export_handoff

HANDOFF_MANIFEST = export_handoff(
    PREPARED,
    OOF_OUTPUT,
    VALIDATION_OUTPUT,
    TEST_OUTPUT,
    VALIDATION_FOLD_OUTPUT,
    TEST_FOLD_OUTPUT,
    ROOT,
    PROJECT_CONFIG,
)
print("Fusion handoff row counts:")
for name, count in HANDOFF_MANIFEST["row_counts"].items():
    print(f"  {name}: {count}")

Fusion handoff row counts:
  member1_oof_predictions.csv: 4226
  member1_validation_predictions.csv: 452
  member1_test_predictions.csv: 453
  member1_train_engineered_features.csv.gz: 4226
  member1_validation_engineered_features.csv.gz: 452
  member1_test_engineered_features.csv.gz: 453


## 9. Evaluation

### Continuous metrics

Because CatBoost is trained as a regressor, continuous metrics are primary:

- **MAE:** average absolute error on the original 1–10 scale. It is easy to interpret and
  less dominated by a few large errors than RMSE.
- **RMSE:** square root of mean squared error. It penalizes large errors more strongly.
- **Pearson correlation:** linear association between true and predicted scores.
- **Spearman correlation:** rank/monotonic association, useful when ordering is correct
  even if the score scale is compressed.

A model can have good correlation but poor calibration, so correlation is never reported
alone.

### Ordered-class metrics

The frozen `clinical_class` is compared with classes obtained from Member 1's validation-
frozen prediction thresholds. The notebook reports:

- accuracy;
- precision, recall, and F1 for every class;
- macro F1, which gives every class equal weight; and
- Very-High PR-AUC, which is useful for a small high-risk class.

Accuracy alone would be misleading because Low is the majority class. Recall asks, “of
the true examples in this class, how many were recovered?” Precision asks, “of the
examples predicted as this class, how many were correct?”

### How to interpret the tabular result

The accepted Member 1 locked-test MAE is about 1.289. The branch is materially weaker than
the final Transformer and fusion systems, and it does not meet all internal class-recall
objectives. That is not a pipeline failure: Member 1 is retained because structured signals
can still contribute complementary information to late fusion.

### Why compare OOF, validation, and test separately?

OOF measures training-set generalization under the supplied fold protocol. Validation is
development evidence. Test is the one-time final estimate. Keeping them separate prevents
a single attractive number from hiding which data were used to make decisions.


In [9]:
from src.evaluate_model import evaluate_all
import pandas as pd

METRICS = evaluate_all(
    PREPARED,
    OOF_OUTPUT,
    VALIDATION_OUTPUT,
    TEST_OUTPUT,
    ROOT,
    PROJECT_CONFIG,
)

continuous_table = pd.DataFrame(METRICS["continuous"]).T
print("Continuous metrics")
display(continuous_table)

for split in ["oof", "validation", "test"]:
    print(f"\n{split.upper()} per-class metrics")
    display(pd.read_csv(OUTPUT_PATHS["metrics"] / f"{split}_per_class_metrics.csv"))
    print("Accuracy:", METRICS["classification"][split]["accuracy"])
    print("Macro F1:", METRICS["classification"][split]["macro_f1"])
    print("Very-High PR-AUC:", METRICS["classification"][split]["very_high_pr_auc"])

Continuous metrics


,mae,rmse,pearson,spearman,prediction_min,prediction_max,prediction_mean
oof,1.266346,1.578170,0.694309,0.691388,1.0,7.820622,3.485157
validation,1.265717,1.592392,0.688642,0.657722,1.0,7.146049,3.058418
test,1.288891,1.655572,0.672657,0.626100,1.0,6.843696,3.110318



OOF per-class metrics


,class,precision,recall,f1,support
0,Low,0.820845,0.645547,0.722718,2257
1,Moderate,0.287950,0.388186,0.330638,948
2,High,0.383520,0.512228,0.438627,736
3,Very High,0.400000,0.266667,0.320000,285


Accuracy: 0.5390440132513015
Macro F1: 0.4529958195740017
Very-High PR-AUC: 0.3407693655871124

VALIDATION per-class metrics


,class,precision,recall,f1,support
0,Low,0.864979,0.690236,0.767790,297
1,Moderate,0.264706,0.537313,0.354680,67
2,High,0.437500,0.444444,0.440945,63
3,Very High,0.733333,0.440000,0.550000,25


Accuracy: 0.6194690265486725
Macro F1: 0.5283537367544284
Very-High PR-AUC: 0.5560359147460519

TEST per-class metrics


,class,precision,recall,f1,support
0,Low,0.875000,0.707071,0.782123,297
1,Moderate,0.169492,0.298507,0.216216,67
2,High,0.422535,0.476190,0.447761,63
3,Very High,0.500000,0.461538,0.480000,26


Accuracy: 0.6004415011037527
Macro F1: 0.4815250788185
Very-High PR-AUC: 0.429152947856119


## 10. Explainability

### What is actually produced in the frozen run?

The delivered executed notebook computes CatBoost's global split-based feature importance
for each fold model and averages the results. The most important features include
starter/reply context, negative emoji/lexical counts, post length, normalized ratios,
sub-category, and account/activity variables.

Native CatBoost SHAP is implemented but disabled by default to keep reproduction fast.
It can be enabled with `MEMBER1_RUN_SHAP=true`. Therefore, the frozen report should claim
the executed **global feature-importance result** and describe native SHAP as optional,
unless a SHAP-enabled run is separately archived.

### Feature importance versus SHAP

- CatBoost's standard global importance summarizes how much features contribute to model
  splits/prediction changes across trees.
- SHAP assigns a local additive contribution to each feature for each example relative to
  a baseline; mean absolute SHAP can then summarize global magnitude.

Both are post-hoc descriptions of the fitted model. Neither proves that a feature causes
stress. Correlated features can divide or redistribute importance, and importance values
should not be interpreted as clinical weights.

### Why explainability is not feature selection here

The final 67-feature contract and model were frozen before this explanation stage.
Removing features because they look unimportant would create a new model and require fresh
validation. The explanation output is therefore for interpretation, debugging, and audit,
not for retroactive tuning.


In [10]:
from src.explain_model import explain_models

EXPLAINABILITY = explain_models(
    FOLD_MODELS,
    PREPARED,
    ROOT,
    PROJECT_CONFIG,
    run_shap=RUN_NATIVE_SHAP,
)
print(EXPLAINABILITY)
print("Explainability is used for interpretation/reporting, not formal feature selection.")
display(__import__('pandas').read_csv(OUTPUT_PATHS["explainability"] / "catboost_global_feature_importance.csv").head(20))
if EXPLAINABILITY["shap_run"]:
    display(__import__('pandas').read_csv(OUTPUT_PATHS["explainability"] / "native_shap_global_ranking.csv").head(20))

{'method': 'CatBoost feature importance', 'feature_count': 67, 'used_for_formal_feature_selection': False, 'shap_run': False}
Explainability is used for interpretation/reporting, not formal feature selection.


,feature,mean_importance,std_importance
0,is_starter,16.586212,0.698059
1,post_neg_emoji,7.932920,0.521598
2,post_char_count,7.260147,0.441922
3,post_neg_count,7.256379,0.546417
4,post_pos_word_ratio,4.940461,0.128436
5,post_temp_neg_ratio,3.738173,0.435595
6,post_word_count,3.707628,0.598588
7,sub_category,3.261338,0.487211
8,post_neg_count_temp,2.282476,0.345393
9,post_neg_word_ratio,2.042093,0.323666


## 11. Artifact verification

A reproducible project is more than a notebook that reaches the last cell. This block
verifies the concrete products required by the scientific contract:

- all five native fold models exist;
- OOF, validation, and test files exist with expected row counts;
- their ID sets exactly match the frozen roles;
- IDs are unique and no role overlaps another;
- predictions are finite and within the configured range;
- holdout ensemble disagreement is present;
- metrics, figures, feature importance, model manifest, and handoff files exist.

These checks detect silent partial runs, stale files, accidental row loss, and misaligned
exports. They do not prove the model is accurate or clinically valid; they prove that the
declared pipeline and its artifacts are complete and internally consistent.


In [11]:
from src.verify_artifacts import verify_artifacts

VERIFICATION = verify_artifacts(ROOT, PROJECT_CONFIG, PREPARED)
print("Artifact verification passed:", VERIFICATION["passed"])
failed = [name for name, passed in VERIFICATION["checks"].items() if not passed]
print("Failed checks:", failed)

Artifact verification passed: True
Failed checks: []


## 12. Lightweight tests

The test suite checks reusable invariants outside notebook state. This is important because
notebooks can hide dependencies in execution order.

The tests cover:

- input-contract requirements and forbidden features;
- prediction schemas, ID coverage, uniqueness, and role separation;
- OOF semantics and holdout ensemble behavior; and
- deterministic feature engineering / reproducibility expectations.

Passing tests means the implementation follows its declared data contract. It does not
replace statistical evaluation, external validation, or human review.


In [12]:
import pytest

TEST_EXIT_CODE = pytest.main(["-q", str(ROOT / "tests")])
if TEST_EXIT_CODE != 0:
    raise RuntimeError(f"Project tests failed with exit code {TEST_EXIT_CODE}")
print("All lightweight project tests passed.")

.

.

.

                                                                      [100%]


3 passed in 0.55s


All lightweight project tests passed.


## 13. Final reproduction summary

The final cell writes a machine-readable summary of:

- role and fold counts;
- OOF/validation/test coverage;
- continuous metrics;
- model/handoff artifact locations;
- future-inference dependencies; and
- known limitations.

### Methodological path demonstrated

The important learned path is:

**frozen leakage-safe data contract → deterministic structured feature engineering →
native categorical/missing-value handling → weighted grouped CatBoost folds → honest OOF
training predictions → five-model holdout ensembles → multi-metric evaluation → post-hoc
explanation → verified Member C handoff.**

This path is more important than merely obtaining a final MAE because it explains why the
result is credible and how it can be reproduced.

### Frozen limitations

1. Structured signals cannot fully represent semantics, negation, or narrative context.
2. The tabular branch does not meet every internal class-recall objective by itself.
3. Some demographic/metadata fields can be missing, imputed, or forum-specific.
4. Exact raw-post inference requires the upstream count/signal extractor.
5. The labeled/evaluation data are safety-enriched and do not estimate natural website
   prevalence or alert workload.
6. The locked test has already been opened; future improvements require a new untouched
   confirmation set.

The appropriate use is as one component of the final human-in-the-loop risk-monitoring
research system, not as an autonomous diagnostic or emergency-decision model.


In [13]:
from pathlib import Path

assert VERIFICATION["passed"]
assert TEST_EXIT_CODE == 0
summary = {
    "model": MODEL_CONFIG["model_name"],
    "full_retrain": FULL_RETRAIN,
    "input_role_counts": AUDIT["role_counts"],
    "fold_counts": AUDIT["fold_counts"],
    "oof_coverage": len(OOF_OUTPUT),
    "validation_coverage": len(VALIDATION_OUTPUT),
    "test_coverage": len(TEST_OUTPUT),
    "oof_metrics": METRICS["continuous"]["oof"],
    "validation_metrics": METRICS["continuous"]["validation"],
    "test_metrics": METRICS["continuous"]["test"],
    "generated_handoff_dir": str((ROOT / "data" / "handoff").relative_to(ROOT)),
    "model_manifest": "models/model_manifest.json",
    "future_inference_required_files": [
        "models/fold_0/model.cbm ... models/fold_4/model.cbm",
        "src/feature_engineering.py or data/processed/member1_preprocessor.joblib",
        "all upstream metadata/count columns documented in member1_preprocessor_config.json",
    ],
    "limitations": [
        "Tabular-only component does not meet every final-fusion recall floor.",
        "Original raw-text positive/negative lexicon extractor was not supplied.",
    ],
}
save_json(summary, OUTPUT_PATHS["metrics"] / "final_notebook_summary.json")
print(json.dumps(summary, indent=2, ensure_ascii=False))
logging.info("Project reproduction completed successfully")
print("\nPROJECT REPRODUCTION COMPLETED SUCCESSFULLY")

{
  "model": "CatBoostRegressor",
  "full_retrain": true,
  "input_role_counts": {
    "train": 4226,
    "embargo": 484,
    "test": 453,
    "validation": 452
  },
  "fold_counts": {
    "0": 844,
    "1": 844,
    "2": 848,
    "3": 847,
    "4": 843
  },
  "oof_coverage": 4226,
  "validation_coverage": 452,
  "test_coverage": 453,
  "oof_metrics": {
    "mae": 1.2663455745663572,
    "rmse": 1.578170360662832,
    "pearson": 0.6943086504982012,
    "spearman": 0.6913879467096139,
    "prediction_min": 1.0,
    "prediction_max": 7.820622180149126,
    "prediction_mean": 3.4851573712246506
  },
  "validation_metrics": {
    "mae": 1.265716936219663,
    "rmse": 1.5923921199644873,
    "pearson": 0.6886423313852909,
    "spearman": 0.6577221688761343,
    "prediction_min": 1.0,
    "prediction_max": 7.146048721896989,
    "prediction_mean": 3.05841772903046
  },
  "test_metrics": {
    "mae": 1.2888909362071534,
    "rmse": 1.655571811605144,
    "pearson": 0.6726567628825295,
    "sp